In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('loan_data.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

In [ ]:
df.describe(include='all').T

In [ ]:
df['loan_status'].value_counts()

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x='loan_status')
plt.show()

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
numeric_df.corr()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f')
plt.show()

In [ ]:
numeric_df.hist(figsize=(14, 10), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
categorical_columns = df.select_dtypes(include='object').columns.tolist()
categorical_columns

In [ ]:
for column in categorical_columns:
    print(column, df[column].unique())

In [ ]:
X = df.drop('loan_status', axis=1)
y = df['loan_status']

if y.dtype == 'object':
    y = y.astype(str).str.strip().str.lower().map({'yes': 1, 'y': 1, 'approved': 1, 'true': 1, '1': 1, 'no': 0, 'n': 0, 'rejected': 0, 'false': 0, '0': 0})
y = y.astype(int)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns
categorical_features = X_train.select_dtypes(exclude=np.number).columns

numeric_transformer = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_transformer = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([('num', numeric_transformer, numeric_features), ('cat', categorical_transformer, categorical_features)])

In [ ]:
model = Pipeline([('preprocessor', preprocessor), ('classifier', LogisticRegression(max_iter=1000))])
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)

print(f'Accuracy = {accuracy:.4f}')
print(f'Precision = {precision:.4f}')
print(f'Recall = {recall:.4f}')
print(f'F1-Score = {f1:.4f}')
print(f'ROC-AUC = {roc_auc:.4f}')

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
results = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred, 'Probability': y_prob})
results.head(10)